# Notebook Mandiri YOLO Bagian A: Menjalankan Detektor Pralatih

**Pengolahan Citra Digital · Semester Ganjil 2026/2027 · Program Studi Teknik Informatika ITERA**

| | |
|---|---|
| **Posisi** | Pengayaan mandiri setelah materi **Minggu ke-11: Segmentasi Citra** |
| **Konsep kuliah yang dipakai** | Perbedaan klasifikasi, deteksi, dan segmentasi (*slide* “Segmentasi Semantik” dan “Pengayaan Mandiri”) |
| **Waktu** | Sekitar 60 menit |
| **Sifat** | Wajib bagi kelompok yang memakai deteksi objek dan bersifat pengayaan bagi kelompok lain. *Notebook* ini tidak dinilai terpisah. Bukti pengerjaan dikumpulkan per kelompok melalui tab Progres di Daftar Kelompok. |
| **Berikutnya** | Bagian B (Minggu ke-12) tentang anotasi data, IoU, dan mAP |

### Tujuan
Setelah menyelesaikan *notebook* ini, Anda mampu:
1. membedakan keluaran **klasifikasi**, **deteksi**, dan **segmentasi** pada citra yang sama,
2. menjalankan model YOLO pralatih dan membaca keluarannya berupa **kotak pembatas** (*bounding box*), **kelas**, dan **tingkat keyakinan** (*confidence*),
3. menjelaskan pengaruh **ambang *confidence*** terhadap jumlah deteksi, dan
4. menilai apakah model pralatih sudah cukup untuk topik tugas besar kelompok atau perlu dilatih ulang (Bagian C).

### Cara memakai
1. Buka *notebook* ini di Google Colab, lalu pilih **File › Save a copy in Drive** agar perubahan Anda tersimpan.
2. Sebaiknya pilih **Runtime › Change runtime type › T4 GPU**. Bagian A juga dapat berjalan di CPU, hanya lebih lambat.
3. Jalankan sel dari atas ke bawah dengan **Shift + Enter**, dan baca penjelasan sebelum setiap sel kode.
4. Kerjakan pertanyaan **Cek Pemahaman**. Pembahasannya tersedia di bawah setiap pertanyaan, tetapi cobalah menjawab terlebih dahulu.

## 0. Persiapan
Sel berikut memasang pustaka **Ultralytics** yang menyediakan model YOLO beserta bobot hasil pelatihan pada *dataset* COCO (80 kelas objek sehari-hari). Bobot model akan diunduh otomatis saat pertama kali dipakai.

In [ ]:
!pip install -q ultralytics

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from ultralytics import YOLO
from ultralytics.utils import ASSETS   # folder citra contoh bawaan Ultralytics

print("GPU tersedia:", torch.cuda.is_available())
print("Citra contoh:", sorted(p.name for p in ASSETS.glob("*.jpg")))

def tampil(daftar, judul, ukuran=(16, 6)):
    """Menampilkan beberapa citra BGR berdampingan."""
    fig, axs = plt.subplots(1, len(daftar), figsize=ukuran)
    axs = np.atleast_1d(axs)
    for ax, im, t in zip(axs, daftar, judul):
        ax.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
        ax.set_title(t)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## 1. Tiga tugas pada satu citra
Ingat kembali *slide* “Pengayaan Mandiri” pada Minggu ke-11.

| Tugas | Pertanyaan yang dijawab | Keluaran |
|---|---|---|
| **Klasifikasi** | *Apa* isi citra ini? | Satu label beserta probabilitasnya untuk seluruh citra |
| **Deteksi** | *Apa* dan *di mana*? | Daftar kotak pembatas, masing-masing dengan kelas dan *confidence* |
| **Segmentasi** | *Piksel mana* milik objek apa? | Masker untuk setiap objek pada tingkat piksel |

YOLO menyediakan ketiga tugas tersebut. Akhiran nama model menunjukkan tugasnya, yaitu `-cls` untuk klasifikasi, tanpa akhiran untuk deteksi, dan `-seg` untuk segmentasi. Huruf `n` menandakan versi *nano*, yaitu versi yang paling kecil dan paling cepat.

In [ ]:
model_cls = YOLO("yolo11n-cls.pt")   # klasifikasi (ImageNet, 1000 kelas)
model_det = YOLO("yolo11n.pt")       # deteksi (COCO, 80 kelas)
model_seg = YOLO("yolo11n-seg.pt")   # segmentasi instans (COCO, 80 kelas)

citra = str(ASSETS / "bus.jpg")
r_cls = model_cls(citra, verbose=False)[0]
r_det = model_det(citra, verbose=False)[0]
r_seg = model_seg(citra, verbose=False)[0]

top1 = r_cls.names[r_cls.probs.top1]
print(f"Klasifikasi: {top1} (probabilitas {r_cls.probs.top1conf:.2f})")
print("Deteksi    :", len(r_det.boxes), "kotak")
print("Segmentasi :", len(r_seg.masks), "masker")

tampil([cv2.imread(citra), r_det.plot(), r_seg.plot()],
       [f"Klasifikasi: {top1}", "Deteksi: kotak + kelas + confidence", "Segmentasi: masker per objek"])

> **Cek Pemahaman 1.** Klasifikasi hanya memberi satu label untuk seluruh citra. Mengapa label itu belum tentu objek yang Anda cari, padahal di dalam citra terdapat beberapa orang?

<details><summary>Pembahasan</summary>

Model klasifikasi menilai citra secara keseluruhan dan memilih satu kelas yang paling dominan. Pada citra ini, kelas yang terpilih adalah kendaraan besar (*minibus*). Model tidak memberi tahu berapa banyak objek yang ada atau di mana letaknya. Jika tugas Anda adalah menghitung atau menemukan lokasi objek, seperti sampah, mobil di parkiran, atau wajah, yang dibutuhkan adalah deteksi atau segmentasi.
</details>

## 2. Membaca keluaran deteksi
Setiap hasil deteksi berisi tiga informasi berikut.
- **Kotak pembatas** dalam format `xyxy`, yaitu (x1, y1, x2, y2) sebagai pojok kiri atas dan pojok kanan bawah dalam satuan piksel. Titik (0, 0) berada di pojok kiri atas citra.
- **Kelas**, berupa nomor kelas COCO beserta namanya.
- ***Confidence***, yaitu tingkat keyakinan model (0 sampai 1) bahwa kotak tersebut memang berisi objek dari kelas itu.

Sel berikut mengubah keluaran model menjadi tabel agar mudah dibaca. Angka yang muncul di komputer Anda bisa sedikit berbeda jika versi model atau pustakanya berbeda.

In [ ]:
b = r_det.boxes
tabel = pd.DataFrame({
    "kelas": [r_det.names[int(c)] for c in b.cls],
    "confidence": b.conf.cpu().numpy().round(2),
    "x1": b.xyxy[:, 0].cpu().numpy().round(), "y1": b.xyxy[:, 1].cpu().numpy().round(),
    "x2": b.xyxy[:, 2].cpu().numpy().round(), "y2": b.xyxy[:, 3].cpu().numpy().round(),
})
tabel["lebar"] = tabel.x2 - tabel.x1
tabel["tinggi"] = tabel.y2 - tabel.y1
print("Ukuran citra (tinggi, lebar):", r_det.orig_shape)
tabel.sort_values("confidence", ascending=False)

Kotak yang sama juga dapat ditulis dalam format lain. Format **`xywhn`** berisi pusat x, pusat y, lebar, dan tinggi yang masing-masing dibagi ukuran citra sehingga bernilai 0 sampai 1. Format inilah yang menjadi **format label YOLO** dan akan Anda pakai saat menganotasi data di Bagian B.

In [ ]:
print("xyxy  (piksel)     :", b.xyxy[0].cpu().numpy().round(1))
print("xywh  (piksel)     :", b.xywh[0].cpu().numpy().round(1))
print("xywhn (ternormalisasi):", b.xywhn[0].cpu().numpy().round(3))

> **Cek Pemahaman 2.** Hitunglah secara manual nilai `xywhn` sebuah kotak dengan x1 = 100, y1 = 50, x2 = 300, dan y2 = 250 pada citra berukuran lebar 800 dan tinggi 500.

<details><summary>Pembahasan</summary>

Pusat x = (100 + 300)/2 = 200, lalu 200/800 = **0,25**. Pusat y = (50 + 250)/2 = 150, lalu 150/500 = **0,30**. Lebar = 200, lalu 200/800 = **0,25**. Tinggi = 200, lalu 200/500 = **0,40**. Jadi, nilai `xywhn` kotak ini adalah 0,25 untuk pusat x, 0,30 untuk pusat y, 0,25 untuk lebar, dan 0,40 untuk tinggi.
</details>

## 3. Pengaruh ambang *confidence*
YOLO hanya menampilkan kotak dengan *confidence* di atas ambang tertentu, yang secara bawaan bernilai **0,25**. Ambang yang rendah menampilkan lebih banyak kotak, termasuk kotak yang salah. Sebaliknya, ambang yang tinggi lebih selektif, tetapi dapat melewatkan objek. Hal ini mirip dengan memilih nilai T pada pengambangan di Minggu ke-11, yaitu tidak ada satu nilai yang selalu benar.

In [ ]:
citra2 = str(ASSETS / "zidane.jpg")
hasil, judul = [], []
for ambang in [0.10, 0.25, 0.50, 0.75]:
    r = model_det(citra2, conf=ambang, verbose=False)[0]
    hasil.append(r.plot())
    judul.append(f"conf ≥ {ambang}: {len(r.boxes)} kotak")
    print(f"conf ≥ {ambang:.2f} → {len(r.boxes)} kotak:", [r.names[int(c)] for c in r.boxes.cls])
tampil(hasil, judul, ukuran=(20, 4))

> **Cek Pemahaman 3.** Pada topik *deteksi parkir kosong*, kesalahan mana yang lebih merugikan, mobil yang tidak terdeteksi atau deteksi mobil palsu? Ke arah mana Anda akan menggeser ambang?

<details><summary>Pembahasan</summary>

Mobil yang tidak terdeteksi membuat sistem melaporkan slot yang terisi sebagai slot kosong sehingga pengemudi diarahkan ke slot yang ternyata penuh. Jika kesalahan ini yang paling merugikan, ambang perlu diturunkan agar lebih banyak mobil tertangkap, dengan konsekuensi jumlah deteksi palsu bertambah. Keseimbangan ini akan diukur dengan presisi dan *recall* pada Minggu ke-12.
</details>

## 4. Menyaring kelas dan menghitung objek
Model COCO mengenal 80 kelas. Argumen `classes` membatasi deteksi pada kelas tertentu, misalnya untuk menghitung jumlah orang. Periksalah terlebih dahulu apakah objek pada topik Anda termasuk kelas COCO.

In [ ]:
nama = model_det.names
print(len(nama), "kelas COCO:")
print(", ".join(f"{i}:{n}" for i, n in nama.items()))

In [ ]:
r_orang = model_det(citra, classes=[0], verbose=False)[0]      # 0 = person
r_kend  = model_det(citra, classes=[2, 5, 7], verbose=False)[0]  # car, bus, truck
print("Jumlah orang    :", len(r_orang.boxes))
print("Jumlah kendaraan:", len(r_kend.boxes))
tampil([r_orang.plot(), r_kend.plot()], ["Hanya kelas orang", "Hanya kendaraan"], ukuran=(12, 6))

| Topik tugas besar | Kelas COCO yang relevan | Cukup dengan model pralatih? |
|---|---|---|
| Deteksi parkir kosong | car, motorcycle, truck | Sebagian. Mobil dapat terdeteksi, tetapi *slot kosong* bukan kelas COCO. |
| Deteksi sampah di kampus | bottle, cup (sebagian) | Tidak. Sebagian besar jenis sampah tidak ada di COCO. |
| Pengenalan makanan kantin | sandwich, pizza, donut, dan lainnya | Tidak. Makanan lokal tidak ada di COCO. |
| Deteksi kantuk dari wajah | person | Tidak. Diperlukan kelas mata terbuka dan mata tertutup. |

Kelas yang tidak ada di COCO memerlukan **data berlabel sendiri** (Bagian B) dan **pelatihan ulang** atau *fine-tuning* (Bagian C).

## 5. Dari kotak ke masker: kaitan dengan segmentasi
Kotak pembatas selalu berbentuk persegi panjang sehingga ikut mencakup sebagian latar. Masker segmentasi hanya mencakup piksel objek. Sel berikut membandingkan luas kotak dan luas masker untuk setiap objek.

In [ ]:
masker = r_seg.masks.data.cpu().numpy()          # (N, h, w) pada resolusi model
H, W = r_seg.orig_shape
baris = []
for i in range(len(r_seg.boxes)):
    x1, y1, x2, y2 = r_seg.boxes.xyxy[i].cpu().numpy()
    luas_kotak = (x2 - x1) * (y2 - y1)
    m = cv2.resize(masker[i], (W, H), interpolation=cv2.INTER_NEAREST)  # samakan ukuran dengan citra asli
    luas_masker = float(m.sum())
    baris.append([r_seg.names[int(r_seg.boxes.cls[i])], round(luas_kotak), round(luas_masker), round(luas_masker / luas_kotak, 2)])
pd.DataFrame(baris, columns=["kelas", "luas kotak (piksel)", "luas masker (piksel)", "rasio masker/kotak"])

> **Cek Pemahaman 4.** Pada tabel di atas, rasio masker terhadap kotak untuk orang berkisar 0,3 sampai 0,6, sedangkan untuk rambu *stop sign* lebih dari 0,8. Mengapa demikian? Kapan topik Anda memerlukan masker, bukan sekadar kotak?

<details><summary>Pembahasan</summary>

Bentuk orang tidak persegi panjang karena ada celah di antara lengan, kaki, dan latar di sekitar kepala sehingga banyak piksel latar ikut masuk ke dalam kotak. Rambu berbentuk segi delapan yang hampir memenuhi kotaknya. Bus memiliki rasio sedang karena sebagian badannya terpotong tepi citra dan tertutup orang. Masker diperlukan jika yang diukur adalah luas atau bentuk objek, misalnya luas daun yang sakit atau penggantian latar foto, bukan sekadar jumlah dan lokasinya.
</details>

## 6. Mencoba citra kelompok Anda
Unggah 3 sampai 5 citra dari topik tugas besar kelompok Anda, lalu jalankan deteksi. Catat kelas yang muncul dan nilailah apakah hasilnya masuk akal.

In [ ]:
from google.colab import files
unggahan = files.upload()            # pilih beberapa citra (jpg/png)
kelas_kelompok = set()
jumlah_citra_kelompok = len(unggahan)

for nama_berkas in unggahan:
    r = model_det(nama_berkas, conf=0.25, verbose=False)[0]
    kelas_kelompok.update(r.names[int(c)] for c in r.boxes.cls)
    print(nama_berkas, "→", [f"{r.names[int(c)]} {p:.2f}" for c, p in zip(r.boxes.cls, r.boxes.conf)])
    tampil([r.plot()], [nama_berkas], ukuran=(8, 6))

Diskusikan hasil pengamatan berikut bersama kelompok. Ringkasannya akan dicetak oleh sel bukti pengerjaan di akhir *notebook*.
1. Apakah objek pada topik Anda terdeteksi oleh model pralatih? Jika ya, dengan kelas apa?
2. Contoh kesalahan yang muncul, misalnya objek terlewat, kelas salah, atau kotak ganda.
3. Keputusan sementara, yaitu apakah model pralatih sudah cukup atau perlu data berlabel dan pelatihan ulang.

## Ringkasan
- **Klasifikasi** memberi satu label per citra, **deteksi** memberi kotak, kelas, dan *confidence*, sedangkan **segmentasi** memberi masker per piksel.
- Kotak YOLO dapat dibaca dalam format `xyxy` (piksel) atau `xywhn` (format label YOLO).
- Ambang *confidence* menukar objek yang terlewat dengan deteksi palsu. Ukurannya, yaitu presisi dan *recall*, dibahas pada Minggu ke-12.
- Model pralatih COCO hanya mengenal 80 kelas. Topik di luar itu memerlukan anotasi (Bagian B) dan pelatihan ulang (Bagian C).

**Bagian B (Minggu ke-12)** membahas cara menganotasi citra kelompok dalam format YOLO, lalu menghitung **IoU** antara kotak prediksi dan kotak benar.

*Referensi:* J. Redmon dkk., “You Only Look Once: Unified, Real-Time Object Detection,” CVPR 2016 · Dokumentasi Ultralytics YOLO, docs.ultralytics.com · T.-Y. Lin dkk., “Microsoft COCO: Common Objects in Context,” ECCV 2014.

## Bukti pengerjaan kelompok
Setelah semua sel di atas dijalankan, isi kelas dan nomor kelompok pada sel berikut, lalu jalankan sel tersebut. Sel ini merangkum hasil *notebook* Anda dalam satu baris. Salin baris itu ke tab **Progres** di Daftar Kelompok pada minggu berjalan.

Bukti ini dikumpulkan satu kali untuk setiap kelompok, tetapi pastikan setiap anggota sudah mencoba menjalankan *notebook* ini sendiri. Dosen dapat menanyakan isi *notebook* kepada anggota mana pun saat presentasi.

In [ ]:
KELAS = "RA"      # isi RA atau RB
KELOMPOK = 0      # isi nomor kelompok Anda

from datetime import datetime
g = globals()
bagian = [f"[Notebook A] {KELAS}-{KELOMPOK:02d}"]
if "r_det" in g and "r_seg" in g and "top1" in g:
    bagian.append(f"bus.jpg: {top1}, {len(r_det.boxes)} kotak, {len(r_seg.masks)} masker")
if "kelas_kelompok" in g and jumlah_citra_kelompok > 0:
    daftar = ", ".join(sorted(kelas_kelompok)) if kelas_kelompok else "tidak ada kelas COCO yang terdeteksi"
    bagian.append(f"citra kelompok: {jumlah_citra_kelompok} citra, kelas terdeteksi: {daftar}")
else:
    bagian.append("citra kelompok: belum dijalankan (Bagian 6)")
bagian.append(datetime.now().strftime("%d-%m-%Y %H:%M"))
if KELOMPOK == 0:
    print("Isi dulu KELAS dan KELOMPOK di atas, lalu jalankan ulang sel ini.")
print(" | ".join(bagian))